# Imports and Load

In [3]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_halving_search_cv
enable_halving_search_cv
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
import xgboost as xgb
import mlflow
import re

mlflow.set_tracking_uri("http://localhost:5000")
print("Tracking URI:", mlflow.get_tracking_uri())

mlflow.set_experiment("customer-churn")
from mlflow.tracking import MlflowClient
client = MlflowClient()


Tracking URI: http://localhost:5000


In [4]:
with open('./data/train.csv', 'r') as f:
    df = pd.read_csv(f)

In [5]:
df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [6]:
numeric_cols = []
categorical_cols = []

for col in df.columns:
    if df[col].dtype in ['object', 'str'] or df[col].nunique() == 2:
        categorical_cols.append(col)
    elif df[col].dtype in ['int64', 'float64']:
        if df[col].nunique() < 6:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)

print(f"Numeric Columns: {numeric_cols}")
print(f"Categorical Columns: {categorical_cols}")

Numeric Columns: ['id', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


---
# 3. Dataset split

In [7]:
df_copy = df.copy()
df_copy = df_copy.drop("id", axis=1)
numeric_cols_ = numeric_cols.copy()
numeric_cols_.remove("id")
categorical_cols_ = categorical_cols.copy()
categorical_cols_.remove("Churn")
X = df_copy.drop("Churn", axis=1)
X = pd.concat([
    X.select_dtypes([], ['object']),
    X.select_dtypes(['object']).apply(pd.Series.astype, dtype='category')
    ], axis=1)


y = df_copy["Churn"].map({"No": 0, "Yes": 1})

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# 4. Feature engineering

In [8]:
X_train.dtypes

SeniorCitizen          int64
tenure                 int64
MonthlyCharges       float64
TotalCharges         float64
gender              category
Partner             category
Dependents          category
PhoneService        category
MultipleLines       category
InternetService     category
OnlineSecurity      category
OnlineBackup        category
DeviceProtection    category
TechSupport         category
StreamingTV         category
StreamingMovies     category
Contract            category
PaperlessBilling    category
PaymentMethod       category
dtype: object

In [9]:
X_train.iloc[[0]]

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
512656,1,13,91.0,1222.65,Male,No,No,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,No,Month-to-month,Yes,Electronic check


# 5. Modeling

## XGBoost with RandomSearchCV

In [12]:
with mlflow.start_run(run_name="random_search_trial") as run:
    mlflow.set_tag("tuning_type", "random-search")
    run_id = run.info.run_id

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        enable_categorical=True,
        scale_pos_weight = 460000 / 133000,
        n_jobs=1
    )

    param_grid = {
        'n_estimators': [200, 400, 600, 800, 100],
        'max_depth': [2, 4, 8, 16, 32],
        'learning_rate': [0.01, 0.1, 0.3],
        'booster': ['gbtree'],
        'subsample': [0.8, 1],
        'colsample_bytree': [0.8, 1],
        'min_child_weight': [1, 3, 5, 7],
        'gamma': [0, 0.1, 0.3, 1],
        'reg_alpha': [0, 0.1, 1, 10],
        'reg_lambda': [0, 0.5, 1, 5, 10],

    }

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=10,
        scoring="roc_auc",
        cv=cv,
        verbose=3,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    print("Best params:")
    print(search.best_params_)

    print("Best mean AUC:")
    print(search.best_score_)
    
    mlflow.log_param("n_estimators", search.best_params_['n_estimators'])
    mlflow.log_param("max_depth", search.best_params_['max_depth'])
    mlflow.log_param("learning_rate", search.best_params_['learning_rate'])
    mlflow.log_param("booster", search.best_params_['booster'])
    mlflow.log_param("subsample", search.best_params_['subsample'])
    mlflow.log_param("colsample_bytree", search.best_params_['colsample_bytree'])
    mlflow.log_metric("roc_auc", search.score(X_val, y_val))
    mlflow.sklearn.log_model(search.best_estimator_, name='grid_xgb_model', input_example=X_train.iloc[[0]], registered_model_name="customer-churn",)

versions = client.search_model_versions(
    f"name='customer-churn' and run_id='{run_id}'"
)
version = versions[0].version

client.set_registered_model_alias(
    name="customer-churn",
    alias="candidate",
    version=version
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params:
{'subsample': 0.8, 'reg_lambda': 5, 'reg_alpha': 1, 'n_estimators': 600, 'min_child_weight': 7, 'max_depth': 2, 'learning_rate': 0.3, 'gamma': 1, 'colsample_bytree': 1, 'booster': 'gbtree'}
Best mean AUC:
0.9153265643030087


2026/03/27 11:33:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
d:\Documents\ds-ml-portfolio\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. S

🏃 View run random_search_trial at: http://localhost:5000/#/experiments/1/runs/65e1945b6a22414cb107ce89a8f58808
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [13]:
def promote_to_stg(version):
    client.set_registered_model_alias(
        name="customer-churn",
        alias="stg",
        version=version
    )


prd_version = None
stg_version = None

try:
    prd_version = client.get_model_version_by_alias("customer-churn", alias="prd")
    prd_run_id = prd_version.run_id
    prd_run = client.get_run(prd_run_id)

    prd_auc = prd_run.data.metrics["roc_auc"]
except Exception as e:
    print(e)

if prd_version is None:
    try:
        stg_version = client.get_model_version_by_alias("customer-churn", alias="stg")
        stg_run_id = stg_version.run_id
        stg_run = client.get_run(stg_run_id)

        stg_auc = stg_run.data.metrics["roc_auc"]
    except Exception as e:
        print(e)


candidate_version = client.get_model_version_by_alias("customer-churn", alias="candidate")
candidate_run_id = candidate_version.run_id
candidate_run = client.get_run(candidate_run_id)

candidate_auc = candidate_run.data.metrics["roc_auc"]

if (stg_version is None and prd_version is None) or (prd_version is None and candidate_auc > stg_auc) or (prd_version is not None and candidate_auc > prd_auc):
    print("Promoting to stage")
    promote_to_stg(version)

INVALID_PARAMETER_VALUE: Registered model alias prd not found.
Promoting to stage
